# An Assistant Answering from Company Documents (RAG)

An LLM doesn't know a company's internal documents: asked about them, it makes up a plausible answer. **Retrieval-augmented generation** (RAG) gives it, along with the question, the relevant excerpts of a document base.

Ingredients of this demonstration, all open and runnable on Colab:

- a small "assistant" LLM, [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)
- a multilingual *embedding* model, which turns a text into a vector to search by meaning

A GPU speeds up the demonstration (`Runtime > Change runtime type`).

In [ ]:
!pip install -q "transformers>=4.56,<5" accelerate sentence-transformers

import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
LLM = "Qwen/Qwen2.5-1.5B-Instruct"
EMBEDDER = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

tokenizer = AutoTokenizer.from_pretrained(LLM)
model = AutoModelForCausalLM.from_pretrained(LLM, dtype="auto").to(device)
embedder = SentenceTransformer(EMBEDDER, device=device)
print(device)

## The Document Base

The (fictional) internal guide of the Acme company: each paragraph becomes an excerpt (*chunk*), prefixed by the title of its section.

In [ ]:
DOCUMENT = """
Remote work
Permanent employees can work remotely up to 2 days per week, with their manager's approval.
Requests are made in the "Kiosk" HR tool, at least 15 days before the first remote day.
Acme provides a screen and an ergonomic chair to employees working remotely at least 1 day per week.

Leave
Each employee has 25 days of paid leave and 11 days off in lieu (RTT) per year.
Leave of more than 5 consecutive days must be booked in Kiosk at least 1 month in advance.

Expense reports
Meals during a business trip are reimbursed up to €23.50 per meal, with a receipt.
Expense reports must be filed in Kiosk within 45 days of the expense.
Reimbursement is paid with the salary of the month following approval.

Business travel
Train is mandatory for trips under 4 hours, in second class.
Beyond that, economy-class flights are allowed, with the department director's approval.
Hotel nights are reimbursed up to €140 in Paris and €95 elsewhere.

IT security
Two-factor authentication is mandatory on all company tools.
Copying customer data into a public AI tool is forbidden; use the internal assistant "AcmeGPT".
Any suspicious email must be forwarded to security@acme.example.

Training
Each employee is entitled to 3 days of training per year, on top of their personal training account.
Training requests are approved during the annual review.
"""

In [ ]:
chunks = []
for section in DOCUMENT.strip().split("\n\n"):
    title, *paragraphs = section.strip().splitlines()
    chunks.extend(f"{title} — {paragraph}" for paragraph in paragraphs)
print(len(chunks))
chunks[:3]

## Without RAG: the Model Makes Things Up

Let's plug the model in as an HR assistant, without giving it the guide, which it has never seen:

In [ ]:
ROLE = "You are Acme's HR assistant. Answer employees' questions precisely and concisely."
QUESTION = "What is the reimbursement cap for a meal during a business trip?"

def ask(messages: list[dict]) -> str:
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=200, do_sample=False,
                                temperature=None, top_p=None, top_k=None)
    prompt_length = inputs["input_ids"].shape[1]
    return tokenizer.decode(output[0, prompt_length:], skip_special_tokens=True)


print(ask([{"role": "system", "content": ROLE}, {"role": "user", "content": QUESTION}]))

The answer looks serious, but it has nothing to do with the guide (which says €23.50): it is a **hallucination**. Depending on the question, the model sometimes admits it doesn't know, but you can't count on it.

## Retrieving the Relevant Excerpts

We compute the vector of every excerpt once and for all. For a question, we keep the excerpts whose vectors are closest (dot product of normalized vectors, between -1 and 1):

In [ ]:
chunk_embeddings = embedder.encode(chunks, normalize_embeddings=True)


def retrieve(question: str, k: int = 3) -> list[tuple[float, str]]:
    question_embedding = embedder.encode(question, normalize_embeddings=True)
    scores = chunk_embeddings @ question_embedding
    best = scores.argsort()[::-1][:k]
    return [(float(scores[i]), chunks[i]) for i in best]


for score, chunk in retrieve(QUESTION):
    print(f"{score:.2f}  {chunk}")

## With RAG: Answering from the Excerpts

We give the model the retrieved excerpts, with the instruction to stick to them and cite its sources:

In [ ]:
INSTRUCTIONS = ("Answer only from the provided excerpts, "
                "citing the number of the excerpt used. If the answer isn't there, "
                "say that you don't know.")
EXCERPTS_LABEL = "Excerpts:"
QUESTION_LABEL = "Question:"

def rag(question: str) -> str:
    excerpts = "\n".join(f"[{i + 1}] {chunk}"
                         for i, (_, chunk) in enumerate(retrieve(question)))
    return ask([
        {"role": "system", "content": f"{ROLE} {INSTRUCTIONS}"},
        {"role": "user", "content": f"{EXCERPTS_LABEL}\n{excerpts}\n\n{QUESTION_LABEL} {question}"},
    ])


print(rag(QUESTION))

## A Question Outside the Base

The guide says nothing about company cars. A good assistant must admit it rather than make something up:

In [ ]:
OUTSIDE_QUESTION = "What is Acme's policy on company cars?"

print(rag(OUTSIDE_QUESTION))

## A Differently Worded Question

The search works by meaning, not by exact words: "lunch on a work trip" finds the excerpt about meals during business trips.

In [ ]:
REPHRASED_QUESTION = "How much can I spend on lunch when I'm on a work trip?"

for score, chunk in retrieve(REPHRASED_QUESTION):
    print(f"{score:.2f}  {chunk}")
print()
print(rag(REPHRASED_QUESTION))

## Takeaways

- Without access to the documents, an LLM hallucinates confidently
- RAG combines a search by meaning with a generation guided by the retrieved excerpts
- Citing sources makes answers checkable
- Quality depends first on the document base (up to date, well split) and on the search: a bad search gives a bad answer
- Even with RAG, a small model can be wrong: evaluate on your own questions before deploying